# analytic filters

this notebook builds cheap physics-based filters that screen rocket designs before they ever reach the KSP simulator.

## context

the structural validator (goal 0) tells us whether a rocket is *possible* — valid parts, connected tree, compatible propellants. it says nothing about whether the rocket can actually fly.

KSP is slow. we can't afford to simulate every candidate design. the analytic filters sit between the validator and KSP, using back-of-envelope physics to reject designs that are obviously incapable of reaching the mission target. only designs that pass all filters get handed to KSP.

## the three filters

**1. thrust-to-weight ratio (TWR)**

$$TWR = \frac{F_{thrust}}{m_{total} \cdot g}$$

if TWR < 1.0 at launch, the rocket cannot lift off. this is a hard physical impossibility — no simulation needed. we'll use a slightly higher threshold (~1.2) to avoid designs that technically lift off but perform poorly.

**2. delta-v estimate**

$$\Delta v = \sum_{\text{stages}} I_{sp} \cdot g_0 \cdot \ln\left(\frac{m_{wet}}{m_{dry}}\right)$$

using the tsiolkovsky rocket equation, applied per stage and summed. if total delta-v is below the mission budget (3400 m/s for low Kerbin orbit), the rocket cannot reach the target regardless of how it's flown.

**3. burn time**

$$t_{burn} = \frac{m_{propellant}}{\dot{m}} = \frac{m_{propellant} \cdot I_{sp} \cdot g_0}{F_{thrust}}$$

a sanity check: does each stage burn for a meaningful duration? a stage that exhausts in 2 seconds is probably a design artifact, not a useful stage.

## what we need from the parts library

all three filters are computable from data we already have:

| quantity | source |
|----------|--------|
| engine thrust (kN) | `part['engine']['max_thrust']` |
| engine Isp (s) | `part['engine']['isp']` |
| part dry mass (t) | `part['mass']` |
| tank resource amounts | `part['resources']` |
| resource mass (t/unit) | scraped from `ResourcesGeneric.cfg` |

## plan

1. scrape KSP resource densities from `ResourcesGeneric.cfg`
2. write `compute_twr(rocket_dict, parts_by_name, resource_lookup)` — total thrust vs total mass
3. write `compute_delta_v(rocket_dict, parts_by_name, resource_lookup)` — per-stage rocket equation, summed
4. write `compute_burn_time(rocket_dict, parts_by_name, resource_lookup)` — per-stage propellant mass / mass flow rate
5. write `filter_rocket(rocket_dict, parts_by_name, resource_lookup, goal)` — runs all three, returns pass/fail with reasons
6. test on the toy rocket and some obviously bad designs
7. extract to `src/filters.py`

parse resources config and make dict of resources, keyed by name and with a value of its density

In [ ]:
from pathlib import Path
from src.scraper import parse_cfg
import json
import numpy as np

### parts library
parts_json_file = Path('../data/parts_library.json')
with open(parts_json_file) as f:
    parts_library = json.load(f)

# parts dictionary by name
parts_by_name = {p['name']: p for  p in parts_library }

# toy example (could have loaded but lazy)
rocket_example = {
    "parts": [
        {"id": "pod_0",  "type": "mk1-3pod", "parent": None},
        {"id": "tank_0", "type": "fuelTank", "parent": "pod_0", "attach_node": "bottom"  },
        {"id": "eng_0",  "type": "liquidEngine", "parent": "tank_0", "attach_node": "bottom"},
    ],
    "stages": {"eng_0": 0}
}

# parts list by name
parts = [part["type"] for part in rocket_example["parts"]]

get kerbal part physics resources (fuel density)

In [ ]:
resources_file = Path('/Users/moss/Library/Application Support/Steam/steamapps/common/Kerbal Space Program/GameData/Squad/Resources/ResourcesGeneric.cfg')
with open(resources_file, encoding='utf-8-sig') as f:
    raw = f.read()
resources_data = parse_cfg(raw)

resource_lookup = {}
for child in resources_data['_children']:
    resource_lookup[child['name']] = {'density':float(child['density'])}

print(resource_lookup)


calculate total mass

In [ ]:
# print(parts_by_name['liquidEngine'].keys())
# print(parts_by_name['liquidEngine']['mass_t'])
# print(parts_by_name['fuelTank'].keys())
# print(parts_by_name['fuelTank']['mass_t'])
# print(type(parts_by_name['fuelTank']['mass_t']))
# print(parts_by_name['liquidEngine']['resources'])
# print(parts_by_name['fuelTank']['resources'])

def get_total_mass(parts_list: list,
                   parts_dict: dict,
                   fuel_lookup: dict):

    total_dry_mass = 0
    for part in parts_list:
        mass = parts_dict[part]['mass_t']
        total_dry_mass += mass

    total_fuel_mass = 0
    for part in parts_list:
        part_entry = parts_dict[part]
        if part_entry['resources'] is not None:
            for resource in part_entry['resources'].keys():
                units = part_entry['resources'][resource]
                conversion = fuel_lookup[resource]['density']
                mass = units * conversion
                total_fuel_mass += mass

    total_mass = total_dry_mass + total_fuel_mass
    return total_mass

get_total_mass(parts, parts_by_name, resource_lookup)

calculate thrust

In [ ]:
# parts_by_name['liquidEngine']['engine']
def calculate_thrust(parts_list: list,
                   parts_dict: dict):

    total_thrust = 0
    for part in parts_list:
        part_entry = parts_dict[part]
        if part_entry['engine'] is not None:
            thrust = part_entry['engine']['max_thrust_kn']
            total_thrust += thrust

    return total_thrust

calculate_thrust(parts, parts_by_name)

In [ ]:
def calculate_twr(parts_list: list,
                  parts_dict: dict,
                  fuel_lookup: dict,
                  g_const: float = 9.80665):

    mass = get_total_mass(parts_list, parts_dict, fuel_lookup)
    thrust = calculate_thrust(parts_list, parts_dict)
    twr = thrust / (mass * g_const)

    return twr

calculate_twr(parts, parts_by_name, resource_lookup)

In [ ]:
# two-stage rocket
# stage 1 fires first (booster): eng_booster burns tank_booster, then decoupler fires and drops booster
# stage 0 fires last (upper):    eng_upper burns tank_upper

staged_rocket = {
    "parts": [
        {"id": "pod_0",        "type": "mk1-3pod",    "parent": None},
        {"id": "tank_upper",   "type": "fuelTank",    "parent": "pod_0",      "attach_node": "bottom"},
        {"id": "eng_upper",    "type": "liquidEngine", "parent": "tank_upper", "attach_node": "bottom"},
        {"id": "decoupler_0",  "type": "Decoupler_1", "parent": "eng_upper",  "attach_node": "bottom"},
        {"id": "tank_booster", "type": "fuelTank",    "parent": "decoupler_0","attach_node": "bottom"},
        {"id": "eng_booster",  "type": "liquidEngine", "parent": "tank_booster","attach_node": "bottom"},
    ],
    "stages": {
        "eng_booster":  1,   # fires first
        "decoupler_0":  1,   # jettisons with booster stage
        "eng_upper":    0,   # fires last
    }
}
parts = [part["type"] for part in staged_rocket["parts"]]
print(staged_rocket)

compute delta v

In [ ]:
import math

def compute_delta_v(rocket_dict: dict,
                    parts_list: list,
                    parts_dict: dict,
                    resource_lookup: dict,
                    g_const: float = 9.80665,
                    verbose: bool = False):

    m_wet = get_total_mass(parts_list, parts_dict, resource_lookup)
    total_dv = 0

    # parts lookup by id
    id_to_type = {p['id']: p['type'] for p in rocket_dict['parts']}
    #parent lookup by id for trace
    id_to_parent = {p['id']: p['parent'] for p in rocket_dict['parts']}
    #child lookup by id for reverse order trace
    id_to_children = {}
    for part in rocket_dict['parts']:
        if part['parent'] is not None:
            if part['parent'] not in id_to_children:
                id_to_children[part['parent']] = []
            id_to_children[part['parent']].append(part['id']) # the above 3 lines could be id_to_children.setdefault(p['parent'], []).append(p['id']), but i did not know this

    for stage_num in sorted(set(rocket_dict['stages'].values()), reverse = True):
        engine_id = None
        decoupler_id = None

        for part_id, stage in rocket_dict['stages'].items():
            if stage == stage_num:
                part_type = id_to_type[part_id]
                if parts_dict[part_type]['engine'] is not None:
                    engine_id = part_id
                if part_type.startswith('Decoupler'):
                    decoupler_id = part_id
        if verbose:
            print(engine_id, decoupler_id)

        if engine_id is None:
            continue

        fuel_mass = 0

        propellants = parts_dict[id_to_type[engine_id]]['engine']['propellants'].keys()
        current = engine_id
        while current is not None:
            current_type = id_to_type[current]
            if current_type.startswith('Decoupler'):
                break
            part_resources = parts_dict[current_type]['resources']
            if part_resources is not None:
                for resource, amount in part_resources.items():
                    if resource in propellants:
                        fuel_mass += amount * resource_lookup[resource]['density']
                        if verbose:
                            print(f'current fuel mass is {fuel_mass}')
            current = id_to_parent[current]

        ### delta_v calculation
        engine_type = id_to_type[engine_id]
        isp = parts_dict[engine_type]['engine']['isp']['vacuum']
        m_dry = m_wet - fuel_mass
        dv = isp * g_const * math.log(m_wet / m_dry)
        total_dv += dv
        if verbose:
            print(f"stage {stage_num}: isp={isp}, m_wet={m_wet:.3f}, m_dry={m_dry:.3f}, dv={dv:.1f}")

        if decoupler_id is not None:
            to_visit = [decoupler_id]
            jettisoned = []
            while to_visit:
                current = to_visit.pop()
                jettisoned.append(current)
                to_visit.extend(id_to_children.get(current, []))

            jettisoned_mass = sum(parts_dict[id_to_type[pid]]['mass_t'] for pid in jettisoned)
            m_wet = m_dry - jettisoned_mass
            if verbose:
                print(f"  jettisoned {jettisoned}, dry mass={jettisoned_mass:.3f}t, new m_wet={m_wet:.3f}")
        else:
            m_wet = m_dry

    return total_dv

compute_delta_v(staged_rocket, parts, parts_by_name, resource_lookup)


compute burn rate

In [ ]:
def compute_burn_time(rocket_dict: dict,
                    parts_list: list,
                    parts_dict: dict,
                    resource_lookup: dict,
                    g_const: float = 9.80665,
                    verbose: bool = False):

    burn_times = {}

    # parts lookup by id
    id_to_type = {p['id']: p['type'] for p in rocket_dict['parts']}
    #parent lookup by id for trace
    id_to_parent = {p['id']: p['parent'] for p in rocket_dict['parts']}

    for stage_num in sorted(set(rocket_dict['stages'].values()), reverse = True):
        engine_id = None

        for part_id, stage in rocket_dict['stages'].items():
            if stage == stage_num:
                part_type = id_to_type[part_id]
                if parts_dict[part_type]['engine'] is not None:
                    engine_id = part_id
        if verbose:
            print(engine_id)

        if engine_id is None:
            continue

        fuel_mass = 0

        propellants = parts_dict[id_to_type[engine_id]]['engine']['propellants'].keys()
        current = engine_id
        while current is not None:
            current_type = id_to_type[current]
            if current_type.startswith('Decoupler'):
                break
            part_resources = parts_dict[current_type]['resources']
            if part_resources is not None:
                for resource, amount in part_resources.items():
                    if resource in propellants:
                        fuel_mass += amount * resource_lookup[resource]['density']
                        if verbose:
                            print(f'current fuel mass is {fuel_mass}')
            current = id_to_parent[current]

        ### burn time calculation
        engine_type = id_to_type[engine_id]
        isp = parts_dict[engine_type]['engine']['isp']['vacuum']

        thrust = parts_dict[id_to_type[engine_id]]['engine']['max_thrust_kn']
        burn_time = fuel_mass * isp * g_const / thrust
        burn_times[stage_num] = burn_time
        if verbose:
            print(f"  stage {stage_num}, fuel ={fuel_mass:.3f}, thrust ={thrust:.3f}, burn_time = {burn_time:.1f}s")

    return burn_times

compute_burn_time(staged_rocket, parts, parts_by_name, resource_lookup)

filter rocket

In [ ]:
DV_THRESHOLDS = {
    'atmosphere': 1800,
    'orbit': 3400,
    'mun_orbit': 4300,
    'mun_landing': 5700
}

def filter_rocket(rocket_dict: dict,
                    parts_list: list,
                    parts_dict: dict,
                    resource_lookup: dict,
                    dv_thresholds: dict,
                    goal: str =  'orbit',
                    g_const: float = 9.80665,
                    verbose: bool = False):

    dv_goal = dv_thresholds[goal]

    twr = calculate_twr(parts_list, parts_dict, resource_lookup)
    dv = compute_delta_v(rocket_dict, parts_list, parts_dict, resource_lookup, g_const = g_const)
    burn_time = compute_burn_time(rocket_dict, parts_list, parts_dict, resource_lookup, g_const = g_const)

    checks = [
          (twr < 1.2,  f"TWR too low: {twr:.2f}"),
          (dv < dv_goal, f"insufficient delta-v: {dv:.0f} m/s")
    ]


    reasons = []
    for condition, message in checks:
          if condition:
              reasons.append(message)
              if verbose:
                  print(f"FAIL: {message}")
    for stage_num, t in burn_time.items():
        if t < 5.0:
            reasons.append(f"stage {stage_num} burn time too short: {t:.1f}s")
    return (len(reasons) == 0, reasons)


filter_rocket(staged_rocket, parts, parts_by_name, resource_lookup, DV_THRESHOLDS, goal = 'mun_landing')

In [ ]:
from src.filters import filter_rocket, DV_THRESHOLDS
filter_rocket(staged_rocket, parts, parts_by_name, resource_lookup, DV_THRESHOLDS, goal = 'orbit')